# archive_02_sarima_solution

Этот ноутбук сохранен как архивный этап поиска решения.
В нем зафиксирован более ранний вариант SARIMA на уровне кластеров магазинов.
Он полезен как исследовательский след, но в основную линию экспериментов уже не входит.


# Baseline v2

## SARIMA-бейзлайн: временной ряд на уровне кластера

В этом подходе мы будем использовать модель, которая популярна у эконометристов и известна уже достаточно давно. *SARIMA* (*Seasonal Autoregression Integrated Moving Average*), то есть модель для временных рядов, которая умеет работать с сезонностью, а также нестационарными временными рядами (I - дифференцирует ряд, чтобы он стал стационарным).

Наша текущая задача сложна тем, что нам сразу нужно строить много временных рядов: потому что динамика продаж у разных категорий разная и они, плюсом ко всему, имеют разные единицы измерения. А нам нужно решение, которое прогнозирует продажи для разных категорий, в разных магазинах сети. Мы можем, конечно использовать 33 модели для средних продаж в рамках категории, но мы уже увидели, что магазины всё-таки отличаются и такой подход, скорее всего, даст большую ошибку. Лучше использовать кластеры для группировки магазинов - внутри кластеров магазины похожи. Также мы заметили, что между магазинами внутри кластера довольно сильная линейная зависимость (высокая корреляция) - следовательно можно попробовать использовать пост-корректировку моделью линейной регрессии.

В текущем ноутбуке мы проверим этот подход. В итоге у нас будет: 18 x 33 = 594 SARIMA модели для каждой комбинации (кластер - категория).

# Служебное

In [ ]:
import numpy as np
import pandas as pd
import logging
import warnings
import os
from concurrent.futures import ProcessPoolExecutor, as_completed
from datetime import timedelta, datetime
import time

from statsmodels.tools.sm_exceptions import ValueWarning
import pmdarima as pm

warnings.filterwarnings(
    "ignore",
    category=ValueWarning,
    message="No frequency information was provided, so inferred frequency .*"
)

logger = logging.getLogger("sarima_cv")
if not logger.handlers:
    logger.setLevel(logging.INFO)
    handler = logging.StreamHandler()
    formatter = logging.Formatter(
        fmt="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
        datefmt="%H:%M:%S"
    )
    handler.setFormatter(formatter)
    logger.addHandler(handler)

# Глобальная переменная для train-данных в воркерах
GLOBAL_TRAIN_DF = None

In [ ]:
train_processed = pd.read_csv("../artifacts/datasets/train_processed.csv", parse_dates=["date"])
test_processed = pd.read_csv("../artifacts/datasets/test_processed.csv", parse_dates=["date"])
zero_sales = pd.read_csv("../artifacts/datasets/zero_sales.csv")

print("train shape:", train_processed.shape)
print("test shape:", test_processed.shape)
print("zero_sales shape:", zero_sales.shape)

In [ ]:
store_means = (
    train_processed
    .groupby(["cluster", "family", "store_nbr"])["sales"]
    .mean()
    .rename("store_mean")
    .reset_index()
)

cluster_means = (
    train_processed
    .groupby(["cluster", "family"])["sales"]
    .mean()
    .rename("cluster_mean")
    .reset_index()
)

store_factors = store_means.merge(
    cluster_means, on=["cluster", "family"], how="left"
)
store_factors["k_factor"] = (
    store_factors["store_mean"] / store_factors["cluster_mean"]
).replace([np.inf, -np.inf], np.nan).fillna(1.0)

In [ ]:
store_factors

In [ ]:
def calculate_metrics(y_true, y_pred):
    """
    Расчёт базовых метрик:
    - RMSLE
    - RMSE
    - MAE
    - WAPE (weighted absolute percentage error)
    """
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    # прогноз не должен быть отрицательным
    y_pred = np.clip(y_pred, 0, None)

    # RMSLE
    rmsle = np.sqrt(np.mean((np.log1p(y_true) - np.log1p(y_pred)) ** 2))

    # RMSE
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))

    # MAE
    mae = np.mean(np.abs(y_true - y_pred))

    # WAPE: суммарная абсолютная ошибка / суммарные фактические продажи
    denom = np.sum(np.abs(y_true))
    if denom == 0:
        wape = np.nan
    else:
        wape = np.sum(np.abs(y_true - y_pred)) / denom

    return {
        "RMSLE": rmsle,
        "RMSE": rmse,
        "MAE": mae,
        "WAPE": wape,
    }


In [ ]:
train_processed.isna().sum()

In [ ]:
test_processed.isna().sum()

In [ ]:
# Приведение типов для train_processed
train_processed["store_nbr"] = train_processed["store_nbr"].astype("int16")
train_processed["cluster"] = train_processed["cluster"].astype("int8")
train_processed["is_payday"] = train_processed["is_payday"].astype("int8")

# если onpromotion в train_processed — счётчик акций, а не float
if np.issubdtype(train_processed["onpromotion"].dtype, np.number) and \
   (train_processed["onpromotion"] % 1 == 0).all():
    train_processed["onpromotion"] = train_processed["onpromotion"].astype("int16")

# числовые фичи в float32
float_cols_train = ["sales", "dcoilwtico", "transactions", "sales_sum"]
for col in float_cols_train:
    if col in train_processed.columns:
        train_processed[col] = train_processed[col].astype("float32")

# Приведение типов для test_processed
test_processed["store_nbr"] = test_processed["store_nbr"].astype("int16")
test_processed["cluster"] = test_processed["cluster"].astype("int8")
test_processed["is_payday"] = test_processed["is_payday"].astype("int8")

# onpromotion в test_processed — int, можно оставить int16
test_processed["onpromotion"] = test_processed["onpromotion"].astype("int16")

# Пересчитаем day_of_week из date, чтобы тип был одинаковый
train_processed["day_of_week"] = train_processed["date"].dt.dayofweek.astype("int8")
test_processed["day_of_week"] = test_processed["date"].dt.dayofweek.astype("int8")

# zero_sales: тоже можно ужать типы
zero_sales["store_nbr"] = zero_sales["store_nbr"].astype("int16")
zero_sales["sales"] = zero_sales["sales"].astype("float32")

In [ ]:
def generate_time_splits(
    df,
    n_splits=3,
    horizon_days=16,
    step_days=30,
    date_col="date",
    last_val_end=None,
):
    """
    Генерирует временные сплиты вида expanding window с фиксированным горизонтом валидации.

    Параметры:
    - df: DataFrame с колонкой дат.
    - n_splits: сколько сплитов строим.
    - horizon_days: длина окна валидации в днях (для Favorita = 16).
    - step_days: на сколько дней сдвигаем валидационное окно назад между сплитами.
    - date_col: имя колонки с датой.
    - last_val_end: последняя дата, которую можно использовать как конец валидации
      (если None — берём максимальную дату из df).

    Возвращает:
    - список словарей с полями:
      {
        "name": str,
        "train_idx": Index,
        "val_idx": Index,
        "train_end": Timestamp,
        "val_start": Timestamp,
        "val_end": Timestamp,
      }
    """
    df = df.sort_values(date_col).copy()
    unique_dates = df[date_col].drop_duplicates().sort_values().to_numpy()

    if last_val_end is None:
        last_val_end = pd.to_datetime(unique_dates[-1])
    else:
        last_val_end = pd.to_datetime(last_val_end)

    splits = []

    for k in range(n_splits):
        # конец и начало валидационного окна
        val_end = last_val_end - pd.Timedelta(days=step_days * k)
        val_start = val_end - pd.Timedelta(days=horizon_days - 1)

        # конец обучающего периода
        train_end = val_start - pd.Timedelta(days=1)

        train_mask = df[date_col] <= train_end
        val_mask = (df[date_col] >= val_start) & (df[date_col] <= val_end)

        train_idx = df.index[train_mask]
        val_idx = df.index[val_mask]

        # если вдруг окно вылезло за границы — пропускаем
        if len(train_idx) == 0 or len(val_idx) == 0:
            continue

        splits.append(
            {
                "name": f"split_{k+1}",
                "train_idx": train_idx,
                "val_idx": val_idx,
                "train_end": train_end,
                "val_start": val_start,
                "val_end": val_end,
            }
        )

    return splits

In [ ]:
splits = generate_time_splits(
    train_processed,
    n_splits=2,
    horizon_days=16,
    step_days=30,
    date_col="date",
)

for s in splits:
    tr = train_processed.loc[s["train_idx"]]
    va = train_processed.loc[s["val_idx"]]
    print(
        s["name"],
        "| train:", tr["date"].min(), "→", tr["date"].max(), "| n =", len(tr),
        "| val:", va["date"].min(), "→", va["date"].max(), "| n =", len(va),
    )

# Продолжаем

In [ ]:
def init_worker(train_df):
    """
    Инициализатор воркера: сохраняем train_df в глобальной переменной,
    чтобы не передавать тяжёлый DataFrame в каждый submit.
    """
    global GLOBAL_TRAIN_DF
    GLOBAL_TRAIN_DF = train_df
    logger = logging.getLogger("sarima_cv")
    logger.info("Worker initialized with GLOBAL_TRAIN_DF")

    
def get_series_for_pair(cluster, family):
    """
    Возвращает временной ряд средних продаж для пары (cluster, family)
    из глобального GLOBAL_TRAIN_DF.

    Ряд:
    - индекс: date (отсортирован)
    - значения: средние daily sales по всем магазинам в этом кластере и категории
    """
    global GLOBAL_TRAIN_DF

    if GLOBAL_TRAIN_DF is None:
        raise RuntimeError("GLOBAL_TRAIN_DF is not initialized in worker")

    sub = GLOBAL_TRAIN_DF[
        (GLOBAL_TRAIN_DF["cluster"] == cluster) &
        (GLOBAL_TRAIN_DF["family"] == family)
    ]

    if sub.empty:
        return pd.Series(dtype="float32")

    ts = (
        sub.groupby("date")["sales"]
        .mean()
        .sort_index()
        .astype("float32")
    )
    return ts

Далее определим две служебные функции для одной серии:
- `fit_sarima_one_series(y_train)` — обучение SARIMA на ряде средних продаж;
- `forecast_sarima(model, horizon)` — прогноз на заданный горизонт (по умолчанию 16 дней).

Пока используем auto_arima с недельной сезонностью (m = 7) и небольшими ограничениями по порядкам, чтобы не взрывать время обучения.
Параметры и регуляризацию можно будет потом подправить, если увидим проблемы со скоростью или качеством.

In [ ]:
def is_series_complex_enough(ts: pd.Series) -> bool:
    """
    Быстрая эвристика: имеет ли смысл использовать SARIMA для этого ряда.
    """
    ts = pd.Series(ts).sort_index().astype("float64")

    # 1. Длина ряда — слишком короткие пропускаем
    if len(ts) < 60:
        return False

    # 2. Доля нулей — очень разреженные ряды лучше отдать простому методу
    zero_share = (ts == 0).mean()
    if zero_share > 0.8:
        return False

    # 3. Почти константные ряды
    if ts.std() < 1e-3:
        return False

    # 4. Простейшая недельная сезонность (по дням недели)
    # если индекс — DatetimeIndex
    if isinstance(ts.index, pd.DatetimeIndex):
        dow_means = ts.groupby(ts.index.dayofweek).mean()
        if len(dow_means) == 7:
            if dow_means.max() - dow_means.min() < 0.1 * ts.mean():
                # колебания по дням недели слабы — нет сильной недельной сезонности
                return False

    return True


def fallback_forecast(ts: pd.Series, horizon: int = 16) -> np.ndarray:
    ts = pd.Series(ts).sort_index().astype("float64")

    if ts.empty:
        return np.zeros(horizon, dtype="float32")

    # работаем с plain numpy-массивом для скорости
    values = ts.to_numpy().tolist()
    y_hat = np.empty(horizon, dtype="float32")

    for h in range(horizon):
        # окно до 30 последних значений (история + уже прогноз)
        window = values[-30:]
        med_30 = float(np.nanmedian(window))  

        # сезонный уровень: неделя назад или последнее
        if len(values) >= 7:
            seasonal_level = values[-7]
        else:
            seasonal_level = values[-1]

        base = 0.5 * med_30 + 0.5 * seasonal_level
        base = 0.0 if not np.isfinite(base) else max(base, 0.0)

        y_hat[h] = base
        values.append(base)

    return y_hat

def fit_sarima_one_series(y_train):
    """
    SARIMA с auto_arima (pmdarima) для одной серии средних продаж.

    Используем сезонность по неделе (m=7) и ограниченный диапазон порядков,
    чтобы не взорвать время на множестве рядов.
    """
    logger = logging.getLogger("sarima_cv")

    y = pd.Series(y_train).astype("float64").fillna(0.0).clip(lower=0.0)

    # порог длины ряда, чтобы модель имела смысл
    if len(y) < 30 or y.sum() == 0:
        logger.info("Series too short or zero-sum, skipping SARIMA")
        return None
    
    # дополнительная проверка на NaN перед auto_arima
    if y.isna().any():
        logger.warning(f"Series has NaN after preprocessing: {y.isna().sum()} values")
        return None
    
    try:
        model = pm.auto_arima(
            y,
            seasonal=True,
            m=7,              # недельная сезонность
            start_p=0, max_p=2,
            start_q=0, max_q=2,
            d=None,           # авто-выбор d
            start_P=0, max_P=1,
            start_Q=0, max_Q=1,
            D=None,           # авто-выбор D
            error_action="ignore",
            suppress_warnings=True,
            stepwise=True,
            max_order=5,
            trace=False,
        )
        logger.info(f"auto_arima fitted: order={model.order}, seasonal_order={model.seasonal_order}")
        return model
    except Exception as e:
        logger.warning(f"auto_arima failed: {e}")
        return None


def forecast_sarima(model, horizon=16):
    """
    Возвращает прогноз на horizon дней вперёд для одной серии.

    Если model == None, возвращает нули длины horizon.
    """
    if model is None:
        return np.zeros(horizon, dtype="float32")

    y_forecast = model.predict(n_periods=horizon)
    y_forecast = np.array(y_forecast, dtype="float32")
    y_forecast = np.clip(y_forecast, 0.0, None)

    return y_forecast

## SARIMA для одной пары (cluster, family) на валидации

Теперь напишем функцию `run_sarima_for_pair`, которая:
- берёт train-часть данных и параметры сплита (val_start, val_end),
- строит временной ряд средних продаж по (cluster, family),
- обучает SARIMA на этом ряде,
- прогнозирует продажи на валидационное окно,
- возвращает DataFrame с датами и предсказаниями.

Позже эту функцию будем вызывать параллельно для всех пар (cluster, family) на каждом временном сплите.

In [ ]:
def run_sarima_for_pair(cluster, family, val_start, val_end):
    """
    Строит SARIMA-прогноз для одной пары (cluster, family) на валидационном окне.

    Логика:
    1) строим ряд ts_train;
    2) если ряд пустой/слишком короткий/нулевой → сразу эвристика;
    3) если ряд "простой" по эвристике → сразу эвристика;
    4) иначе пробуем SARIMA, при ошибке → эвристика.
    """
    logger = logging.getLogger("sarima_cv")
    pid = os.getpid()

    ts_train = get_series_for_pair(cluster, family)

    horizon = (val_end - val_start).days + 1
    dates = pd.date_range(val_start, val_end, freq="D")

    # случай пустого / очень короткого / нулевого ряда
    if ts_train.empty or len(ts_train) < 10 or ts_train.sum() == 0:
        logger.info(
            f"fallback (zeros/heuristic) for (cluster={cluster}, family={family}): "
            f"empty/short/zero series"
        )
        y_hat = fallback_forecast(ts_train, horizon=horizon)
        return pd.DataFrame({
            "date": dates,
            "cluster": cluster,
            "family": family,
            "y_pred": y_hat.astype("float32"),
        })

    # эвристика «стоит ли вообще учить SARIMA»
    if not is_series_complex_enough(ts_train):
        logger.info(
            f"skip SARIMA for (cluster={cluster}, family={family}): "
            f"simple series, using heuristic forecast"
        )
        y_hat = fallback_forecast(ts_train, horizon=horizon)
        return pd.DataFrame({
            "date": dates,
            "cluster": cluster,
            "family": family,
            "y_pred": y_hat.astype("float32"),
        })

    logger.info(f"[PID={pid}] start pair (cluster={cluster}, family={family})")

    # замеряем ЧИСТОЕ время fit+forecast
    t0 = time.time()
    try:
        model = fit_sarima_one_series(ts_train)
        y_hat = forecast_sarima(model, horizon=horizon)

        elapsed = time.time() - t0
        logger.info(
            f"[PID={pid}] model trained OK for (cluster={cluster}, family={family}), "
            f"horizon={horizon}, fit+forecast={elapsed:.2f} seconds"
        )
    except Exception as e:
        logger.warning(
            f"run_sarima_for_pair failed for (cluster={cluster}, family={family}): {e}. "
            f"Using heuristic fallback."
        )
        y_hat = fallback_forecast(ts_train, horizon=horizon)

    return pd.DataFrame({
        "date": dates,
        "cluster": cluster,
        "family": family,
        "y_pred": y_hat.astype("float32"),
    })

## Параллельный прогон SARIMA по одному временном сплиту

Теперь соберём всё вместе для одного сплита:
- берём train/val-части по датам;
- составляем список всех пар (cluster, family), которые встречаются в train;
- для каждой пары запускаем `run_sarima_for_pair` (параллельно);
- склеиваем все предсказания в один DataFrame и считаем метрики на уровне всех наблюдений валидации.

Дальше эту же функцию будем использовать в цикле по всем нашим сплитам.

In [ ]:
def predict_sarima_on_split(
    train_df,
    val_df,
    split_name,
    max_workers=1,
    store_factors=None,
    pairs_subset=None,
):
    """
    Строит SARIMA-прогноз для всех пар (cluster, family) на заданном сплите.

    Логика:
    1) Параллельно считаем прогнозы на уровне (date, cluster, family).
    2) Считаем кластерные метрики по средним продажам (cluster, family).
    3) Если передан store_factors, делаем top-down раскладку на магазины
       и считаем метрики на уровне (store_nbr, family, date).

    Параметры:
    - train_df: DataFrame train-части сплита
    - val_df:   DataFrame validation-части сплита
    - split_name: имя сплита для логов
    - max_workers: число воркеров ProcessPoolExecutor
    - store_factors: DataFrame с колонками
        ["cluster", "family", "store_nbr", "k_factor"],
      где k_factor — исторический коэффициент доли магазина
      в средних продажах кластера.
    """
    logger = logging.getLogger("sarima_cv")

    # время старта сплита
    split_start_time = time.time()
    split_start_dt = datetime.now()
    logger.info(
        f"[{split_name}] Start SARIMA prediction on split "
        f"at {split_start_dt.strftime('%Y-%m-%d %H:%M:%S')}"
    )

    val_start = val_df["date"].min()
    val_end = val_df["date"].max()
    logger.info(
        f"[{split_name}] Validation window: "
        f"{val_start.date()} → {val_end.date()}"
    )

    # Список уникальных пар (cluster, family) в train
    pairs = (
        train_df[["cluster", "family"]]
        .drop_duplicates()
        .itertuples(index=False, name=None)
    )
    pairs = list(pairs)
    
    # если передан pairs_subset, фильтруем пары
    if pairs_subset is not None:
        pairs = [p for p in pairs if p in set(pairs_subset)]
        
    n_pairs = len(pairs)
    logger.info(
        f"[{split_name}] Number of (cluster, family) pairs: {n_pairs}"
    )

    results = []
    n_ok = 0
    n_fail = 0

    logger.info(
        f"[{split_name}] Launching ProcessPoolExecutor "
        f"with max_workers={max_workers}"
    )

    # Для оценки среднего времени обработки по паре
    total_pair_time = 0.0

    # Инициализируем воркеры, передав им train_df один раз
    with ProcessPoolExecutor(
        max_workers=max_workers,
        initializer=init_worker,
        initargs=(train_df,),
    ) as executor:
        future_to_meta = {}
        for (cluster, family) in pairs:
            submit_time = time.time()
            fut = executor.submit(
                run_sarima_for_pair, cluster, family, val_start, val_end
            )
            future_to_meta[fut] = (cluster, family, submit_time)

        for i, future in enumerate(as_completed(future_to_meta), start=1):
            cluster, family, submit_time = future_to_meta[future]
            pair_start_time = submit_time

            try:
                df_pred = future.result()
                n_ok += 1
                pair_elapsed = time.time() - pair_start_time
                total_pair_time += pair_elapsed
                logger.info(
                    f"[{split_name}] Pair (cluster={cluster}, family={family}) "
                    f"finished in {pair_elapsed:.2f} seconds "
                    f"(queue + fit+forecast)"
                )
            except Exception as exc:
                n_fail += 1
                logger.warning(
                    f"[{split_name}] Failed pair (cluster={cluster}, "
                    f"family={family}): {exc}"
                )
                pair_elapsed = time.time() - pair_start_time
                total_pair_time += pair_elapsed
                logger.info(
                    f"[{split_name}] Failed pair (cluster={cluster}, "
                    f"family={family}) took {pair_elapsed:.2f} seconds "
                    f"before failure"
                )

                dates = pd.date_range(val_start, val_end, freq="D")
                df_pred = pd.DataFrame({
                    "date": dates,
                    "cluster": cluster,
                    "family": family,
                    "y_pred": np.zeros(len(dates), dtype="float32"),
                })

            results.append(df_pred)

            if i % 20 == 0 or i == n_pairs:
                avg_pair_time = total_pair_time / i
                logger.info(
                    f"[{split_name}] Progress: {i}/{n_pairs} pairs processed "
                    f"(ok={n_ok}, fail={n_fail}), "
                    f"avg pair time so far = {avg_pair_time:.2f} seconds "
                    f"(queue + fit+forecast)"
                )

    # Лог по всему сплиту
    split_elapsed = time.time() - split_start_time
    avg_pair_time_overall = total_pair_time / max(n_pairs, 1)
    logger.info(
        f"[{split_name}] All pairs processed in {split_elapsed:.2f} seconds "
        f"(avg per pair = {avg_pair_time_overall:.2f} seconds, "
        f"queue + fit+forecast)"
    )

    logger.info(f"[{split_name}] All pairs processed. Concatenating predictions...")

    preds_df = pd.concat(results, ignore_index=True)
    logger.info(f"[{split_name}] preds_df shape: {preds_df.shape}")

    # === Кластерный уровень: средние продажи по (date, cluster, family) ===
    val_agg = (
        val_df.groupby(["date", "cluster", "family"])["sales"]
        .mean()
        .reset_index()
        .rename(columns={"sales": "y_true"})
    )
    logger.info(f"[{split_name}] val_agg shape: {val_agg.shape}")

    val_pred_df = preds_df.merge(
        val_agg,
        on=["date", "cluster", "family"],
        how="left",
    )

    before_filter = len(val_pred_df)
    val_pred_df = val_pred_df[~val_pred_df["y_true"].isna()].copy()
    after_filter = len(val_pred_df)
    logger.info(
        f"[{split_name}] val_pred_df filtered: {before_filter} → "
        f"{after_filter} rows with y_true"
    )

    y_true_cluster = val_pred_df["y_true"].values
    y_pred_cluster = val_pred_df["y_pred"].values

    metrics_cluster = calculate_metrics(y_true_cluster, y_pred_cluster)
    metrics_cluster = {f"cluster_{k}": v for k, v in metrics_cluster.items()}

    logger.info(
        f"[{split_name}] Cluster-level metrics: "
        f"RMSLE={metrics_cluster['cluster_RMSLE']:.4f}, "
        f"RMSE={metrics_cluster['cluster_RMSE']:.2f}, "
        f"MAE={metrics_cluster['cluster_MAE']:.2f}, "
        f"WAPE={metrics_cluster['cluster_WAPE']:.4f}"
    )

    # === Store-level: top-down раскладка на магазины (если есть коэффициенты) ===
    metrics_store = {}
    val_pred_store = None
    
    if store_factors is not None:
        # магазины, которые присутствуют в train на этом сплите
        stores_cf = (
            train_df[["cluster", "family", "store_nbr"]]
            .drop_duplicates()
        )

        preds_cf = preds_df.merge(
            stores_cf, on=["cluster", "family"], how="left"
        )

        preds_cf = preds_cf.merge(
            store_factors[["cluster", "family", "store_nbr", "k_factor"]],
            on=["cluster", "family", "store_nbr"],
            how="left",
        )
        preds_cf["k_factor"] = preds_cf["k_factor"].fillna(1.0)

        preds_cf["y_store_pred"] = preds_cf["y_pred"] * preds_cf["k_factor"]

        # реальные продажи на уровне магазина для валидации
        val_store = (
            val_df[["date", "cluster", "family", "store_nbr", "sales"]]
            .rename(columns={"sales": "y_true_store"})
        )

        val_pred_store = val_store.merge(
            preds_cf[[
                "date", "cluster", "family", "store_nbr", "y_store_pred"
            ]],
            on=["date", "cluster", "family", "store_nbr"],
            how="left",
        )

        val_pred_store["y_store_pred"] = (
            val_pred_store["y_store_pred"].fillna(0.0)
        )

        y_true_store = val_pred_store["y_true_store"].values
        y_pred_store = val_pred_store["y_store_pred"].values

        metrics_store = calculate_metrics(y_true_store, y_pred_store)
        metrics_store = {f"store_{k}": v for k, v in metrics_store.items()}

        logger.info(
            f"[{split_name}] Store-level metrics: "
            f"RMSLE={metrics_store['store_RMSLE']:.4f}, "
            f"RMSE={metrics_store['store_RMSE']:.2f}, "
            f"MAE={metrics_store['store_MAE']:.2f}, "
            f"WAPE={metrics_store['store_WAPE']:.4f}"
        )

    # объединяем метрики
    metrics = {}
    metrics.update(metrics_cluster)
    metrics.update(metrics_store)

    metrics["split"] = split_name
    metrics["model"] = "sarima_cluster_family_mean_v1"

    return metrics, val_pred_df, val_pred_store

In [ ]:
s = splits[0]  # split_3
train_df = train_processed.loc[s["train_idx"]]
val_df = train_processed.loc[s["val_idx"]]

# берём первые 80 пар из train сплита
pairs_all = (
    train_df[["cluster", "family"]]
    .drop_duplicates()
    .itertuples(index=False, name=None)
)
pairs_all = list(pairs_all)
pairs_80 = pairs_all[:80]

metrics_s3_small, val_pred_s3_small, val_pred_store_s3_small = predict_sarima_on_split(
    train_df,
    val_df,
    split_name=s["name"] + "_80pairs",
    max_workers=3,
    store_factors=store_factors,
    pairs_subset=pairs_80,
)

metrics_s3_small
val_pred_s3_small.head()
val_pred_store_s3_small.head()

In [ ]:
metrics_s3_small